We start with concatanating two datasets obtained from two reviews and reorder them so that we can manually check if same entries in those papers do not match:

In [8]:
import pandas as pd
from rdkit import Chem

# load data
df1 = pd.read_csv('../data/Gd/reviews/dioury2014.csv')
df2 = pd.read_csv('../data/Gd/reviews/uzal2022.csv')

# filter columns
cols = ['ID', 'smiles', 'doi', 'lgK']
df1 = df1.loc[:,cols]
df2 = df2.loc[:,cols]

# add source info
df1.insert(3, 'source', 'dioury2014')
df2.insert(3, 'source', 'uzal2022')

# merge
df = pd.concat([df1, df2], ignore_index = True).rename(columns = {'ID': 'sourceID'})

# add compound IDs
IDs = {smi: f'rev_{i+1:03d}' for i, smi in enumerate(df.smiles.drop_duplicates())}
df.insert(0, 'ID', [IDs[smi] for smi in df.smiles])

# sort
df = df.sort_values(by = ['ID', 'doi', 'source', 'lgK'])

# view table
pd.set_option('display.max_rows', 400)
df.loc[:,['ID','doi','source','sourceID','lgK']]

,ID,doi,source,sourceID,lgK
0,rev_001,10.1002/ejoc.200400264,dioury2014,1,9.34
361,rev_001,10.1002/ejoc.200400264,uzal2022,B43,9.34
1,rev_001,10.1021/ic960794b,dioury2014,1,12.50
2,rev_002,10.1021/ic960794b,dioury2014,2,19.30
352,rev_003,10.1002/chem.201705528,uzal2022,B34,18.28
3,rev_003,10.1021/ic0608750,dioury2014,3,20.39
351,rev_003,10.1021/ic0608750,uzal2022,B34,20.39
4,rev_003,10.1021/ic960794b,dioury2014,3,21.00
5,rev_004,10.1021/bc8004914,dioury2014,4,19.42
353,rev_004,10.1021/bc8004914,uzal2022,B35,19.42


As a result of manual check we conclude that we can just drop repeating (SMILES, DOI) entries ignoring their source (one of two reviews):

In [ ]:
pd.set_option('display.max_rows', 20)

# drop unwanted data
df = df.drop_duplicates(['ID', 'doi']).reset_index(drop = True)
df

,ID,sourceID,smiles,doi,source,lgK
0,rev_001,1,O=C(O)CN1CCCN(CC(=O)O)Cc2cccc(n2)CN(CC(=O)O)CCC1,10.1002/ejoc.200400264,dioury2014,9.34
1,rev_001,1,O=C(O)CN1CCCN(CC(=O)O)Cc2cccc(n2)CN(CC(=O)O)CCC1,10.1021/ic960794b,dioury2014,12.50
2,rev_002,2,O=C(O)CN1CCCN(CC(=O)O)Cc2cccc(n2)CN(CC(=O)O)CC1,10.1021/ic960794b,dioury2014,19.30
352,rev_003,B34,O=C(O)CN1CCN(CC(=O)O)Cc2cccc(n2)CN(CC(=O)O)CC1,10.1002/chem.201705528,uzal2022,18.28
3,rev_003,3,O=C(O)CN1CCN(CC(=O)O)Cc2cccc(n2)CN(CC(=O)O)CC1,10.1021/ic0608750,dioury2014,20.39
...,...,...,...,...,...,...
388,rev_223,T8,O=C(O)CN(CCOCCN(CC(=O)O)Cc1cccc(C(=O)O)n1)Cc1c...,10.1021/acs.inorgchem.0c00372,uzal2022,20.50
389,rev_224,T9,O=P(O)(O)CN1CCNCCNCCN(CP(=O)(O)O)CC1,10.3390/molecules24183324,uzal2022,19.15
390,rev_225,T10,O=C(O)CN1CCN(CC(=O)O)CCN(C[C@@H](O)C(=O)O)CCN(...,10.1021/acs.inorgchem.1c01927,uzal2022,19.26
391,rev_226,T11,O=C(O)CN1Cc2cccc(n2)CN(CC(=O)O)Cc2cccc(n2)CN(C...,10.1021/acs.inorgchem.9b03345,uzal2022,18.30


In [11]:
df = df.dropna()

df_shuffled = df.sample(frac=1, random_state=42)
df_unique_random = df_shuffled.drop_duplicates(subset=['smiles']).reset_index(drop=True) 
df_unique_random

,ID,sourceID,smiles,doi,source,lgK
0,rev_081,81,CNC(=O)CN(CCN(CCN(CC(=O)O)CC(=O)NC)CC(=O)O)CC(...,10.1016/0730-725x(90)90055-7,dioury2014,16.85
1,rev_022,22,O=C(O)CN1CCN(CC(=O)O)CCN(CC(=O)O)CCN(CC(=O)O)CC1,10.1016/0020-1693(96)05182-1,dioury2014,24.67
2,rev_142,142,O=C(O)CN(CCN(CCN(CC(=O)O)CC(=O)O)CC(=O)O)CCN(C...,10.1016/0223-5234(88)90094-3,dioury2014,28.40
3,rev_035,35,CC(CO)N1CCN(CC(=O)O)CCN(CC(=O)O)CCN(CC(=O)O)CC1,10.1021/ic00095a028,dioury2014,23.90
4,rev_170,A39,O=C(O)CN(CC(=O)O)Cc1cccc(CN(CC(=O)O)CC(=O)O)n1,10.1039/b817343e,uzal2022,18.60
...,...,...,...,...,...,...
212,rev_136,136,CN(CC(O)C(O)C(O)C(O)CO)C(=O)C(COCc1ccccc1)N1CC...,10.1021/ic00038a023,dioury2014,26.40
213,rev_074,74,CCN(CCN(CC(=O)O)CC(=O)O)CCN(CC(=O)O)CC(=O)O,10.1021/ic00212a005,dioury2014,17.79
214,rev_227,T12,CCCCNC(=O)CN(CCN(CCN(CC(=O)O)CC(=O)O)CC(=O)O)C...,10.1039/C7DT04104G,uzal2022,18.78
215,rev_062,62,Nc1ccc(CC(C(=O)O)N(CCN(CC(=O)O)CC(=O)O)CCN(CC(...,10.1021/jm9602118,dioury2014,21.99


In [13]:
# save data
df_unique_random.to_csv('../data/Gd/final/merged_reviews_clean.csv', index = None)